This notebook explores and analyses my YouTube watch history using Python and pandas. It covers parsing raw Google Takeout data, cleaning and structuring watch records, and performing exploratory data analysis with a focus on music listening habits

# Parsing Watch History

## Initiating watch history retrieval

In [4]:
from pathlib import Path
from bs4 import BeautifulSoup


project_root = Path.cwd().parent
watch_path = project_root / "data" / "raw" / "watch-history.html"

print(watch_path)
print(watch_path.exists())

C:\Users\MsnSi\PycharmProjects\Youtube_Music_Analysis\data\raw\watch-history.html
True


## Read data from watch history

Beautiful Soup is a Python library for parsing HTML/XML and turning it into something Python can navigate. Line 1 (htm) loads the html file into one string, and line 2 (soup) is used to parse through that string.


cards
│
├── cards[0] ── Watch history entry #1
├── cards[1] ── Watch history entry #2
├── cards[2] ── Watch history entry #3
├── cards[3] ── Watch history entry #4
└── ...

In [5]:
html = watch_path.read_text(encoding="utf-8")
soup = BeautifulSoup(html, "lxml")

cards = soup.select("div.outer-cell")

print(f"Number of activity records: {len(cards)}")

Number of activity records: 12200


## Grabbing the data from watch history


In [6]:
first_card = cards[0]
content = first_card.select_one("div.content-cell")

list(content.stripped_strings)

['Watched',
 'Mario Kart 8 Deluxe Players are Going Insane right now...',
 'Shortcat',
 '7 Sept 2026, 13:51:45 IST']

## Converting the Watched Video Data records into a dictionary data structure

In [7]:
parts = list(content.stripped_strings)
links = content.select("a")


record = {
    "action": parts[0],
    "title": parts[1],
    "channel": parts[2],
    "watched_at": parts[3],
    "video_url": links[0].get("href"),
}

print(record)

{'action': 'Watched', 'title': 'Mario Kart 8 Deluxe Players are Going Insane right now...', 'channel': 'Shortcat', 'watched_at': '7 Sept 2026, 13:51:45 IST', 'video_url': 'https://www.youtube.com/watch?v=TOKdln0vkrw'}


In [8]:
## Creating all the Records from Watch history

Create empty records list

FOR every card:
    Find the content inside this card

    Extract the strings
    Extract the links

    Build a dictionary

    Add dictionary to records

Finished → records now contains every watch-history entry

In [15]:
records = []
record_number = 0
bad_records = 0

for card in cards:
    content = card.select_one("div.content-cell")
    parts = list(content.stripped_strings)
    links = content.select("a")

    if len(parts) < 4:
        bad_records += 1
        print("PARTS:", parts)
        print("LINKS:", links)
        print("--------------------")
    else:
        record_number += 1

    #record = {
    #    "action": parts[0],
    #    "title": parts[1],
     #   "channel": parts[2],
      #  "watched_at": parts[3],
       # "video_url": links[0].get("href"),
    #}

    #records.append(record)
print("Number of normal records", record_number)
print("Number of bad records: ", bad_records)


PARTS: ['Watched', 'INH CARDS CUSTOMCARDS 3183db07 AIAvatar FaceUGC 9x16 18s A1 NA NL NL Nov2025 VO  1', '4 Sept 2026, 21:23:10 IST']
LINKS: [<a href="https://www.youtube.com/watch?v=J5wkIKQfW_I">INH CARDS CUSTOMCARDS 3183db07 AIAvatar FaceUGC 9x16 18s A1 NA NL NL Nov2025 VO  1</a>]
--------------------
PARTS: ['Watched', 'INFNANO LIFESTYLE BUDGETING 698ef62a Ivelina2UGC UGC 9x16 NA A1 NA BG BG Jul2026 VO', '3 Sept 2026, 23:15:10 IST']
LINKS: [<a href="https://www.youtube.com/watch?v=KkFMhhuYruM">INFNANO LIFESTYLE BUDGETING 698ef62a Ivelina2UGC UGC 9x16 NA A1 NA BG BG Jul2026 VO</a>]
--------------------
PARTS: ['Viewed a post that is no longer available', '1 Sept 2026, 03:39:29 IST']
LINKS: []
--------------------
PARTS: ['Viewed a post that is no longer available', '31 Aug 2026, 14:24:55 IST']
LINKS: []
--------------------
PARTS: ['Viewed a post that is no longer available', '29 Aug 2026, 22:38:37 IST']
LINKS: []
--------------------
PARTS: ['Watched', 'https://www.youtube.com/watch

It seems like our parser is a bit flawed due to some bad data within the watch history, there seems to be 4 types of outliers:

1. Ads: 3 Parts, but no obvious link
2. Community Posts: 3 parts, no link
3. Unlisted Videos: 3 Parts, Link exists in the *links* field
4. Privated Videos: 3 Parts, 1 Link in parts, 2 links in *links field


## Updated Record Retrieval


In [40]:
records = []
record_number = 0
missing_channel_records = 0
non_video_records = 0

zero_links = 0
one_link = 0
two_or_more_links = 0


for card in cards:
    content = card.select_one("div.content-cell")
    parts = list(content.stripped_strings)
    links = content.select("a")

    if parts[0] != 'Watched':
        non_video_records += 1
        continue

    if len(parts) == 4:
        record_number += 1
        record = {
            "action": parts[0],
            "title": parts[1],
            "channel": parts[2],
            "watched_at": parts[3],
            "video_url": links[0].get("href"),
        }

    elif len(parts) == 3:
        missing_channel_records += 1

        if len(links) <= 0:
            zero_links += 1
        elif len(links) == 1:
            one_link += 1
        elif len(links) == 2:
            two_or_more_links += 1

        record = {
            "action": parts[0],
            "title": parts[1],
            "channel": None,
            "watched_at": parts[2],
            "video_url": links[0].get("href"),
        }


    records.append(record)

print("Number of normal records", record_number)
print("Number of records with missing channels: ", missing_channel_records)
print("Number of non-video records: ", non_video_records)

print("Number of bad records with zero links: ", zero_links)
print("Number of records with one link: ", one_link)
print("Number of records with two or more links: ", two_or_more_links)

Number of normal records 9945
Number of records with missing channels:  115
Number of non-video records:  2140
Number of bad records with zero links:  0
Number of records with one link:  115
Number of records with two or more links:  0


In [41]:
zero_links = 0
one_link = 0
two_or_more_links = 0

# Using Pandas

## Implementing the Pandas Library

In [42]:
import pandas as pd

records_dataframe = pd.DataFrame(records)

records_dataframe.head()

,action,title,channel,watched_at,video_url
0,Watched,Mario Kart 8 Deluxe Players are Going Insane r...,Shortcat,"7 Sept 2026, 13:51:45 IST",https://www.youtube.com/watch?v=TOKdln0vkrw
1,Watched,Oops! Nintendo reveals Mario Kart 8's DEBUG mode,Beta64,"7 Sept 2026, 13:50:46 IST",https://www.youtube.com/watch?v=Lg1QI8ZkPRU
2,Watched,I Love These Uma Musume,Starfang,"7 Sept 2026, 13:44:34 IST",https://www.youtube.com/watch?v=5OOE4WORb5Q
3,Watched,第三方居然出了MG飛昇自由？大砲還有拉伸結構，還可以全彈髮射！ #gundam #갓건담 #...,玩具測評總動員,"7 Sept 2026, 13:43:43 IST",https://www.youtube.com/watch?v=R9922GyZmvw
4,Watched,【#なつめぇ誕生日2026】🎂ケーキ食べようよ！【来栖夏芽/にじさんじ】,来栖 夏芽-kurusu natsume-【にじさんじ】,"7 Sept 2026, 13:35:14 IST",https://www.youtube.com/watch?v=Ftji2LKqVXs


## Exploring the properties of the DataFrame

We can use the shape attribute to understand the general structure of the DataFrame

In [43]:
records_dataframe.shape


(10060, 5)

in the below code, we get (100060,5) which likely represents the number of row entries and the number of coloumn entries in our DataFrame

The dtypes attribute is used to expose the types of each column entry in the dataFrame

In [44]:
records_dataframe.dtypes


action        str
title         str
channel       str
watched_at    str
video_url     str
dtype: object

Before changing the data, We already know there should be at least 115 missing channel values from our parsing work, we need to figure out what's missing from those entries.

We can use the isna() function, which will scan through each cell in the Data Frame and check if a value exists, combine this with .sum() to add the up these entries together to see how many of each value is missing.

Previously both channel and video_url had shown 115 entries, this was a mistake as a result of a flawed parser which failed to catch the fact that I failed to realise that just because an entry might have less parts (due to not having a channel), we could still grab their link as that was from another array from the content entry

In [46]:
records_dataframe.isna().sum()

action          0
title           0
channel       115
watched_at      0
video_url       0
dtype: int64

# Cleaning up the data

We need to clean up the data so we can use it for analysis, firstly we'll convert the "watched_at" field into a timestamp so it can be used later for analysis

## Getting Rid of Bad Entries

Getting Rid of Bad Entries

In [90]:
valid_channel_records = records_dataframe[records_dataframe["channel"].notna()]

records_dataframe = valid_channel_records


#print(records_dataframe.shape)
#records_dataframe.head()


records_dataframe.reset_index(drop = True)

,action,title,channel,watched_at,video_url
0,Watched,Mario Kart 8 Deluxe Players are Going Insane r...,Shortcat,2026-09-07 13:51:45,https://www.youtube.com/watch?v=TOKdln0vkrw
1,Watched,Oops! Nintendo reveals Mario Kart 8's DEBUG mode,Beta64,2026-09-07 13:50:46,https://www.youtube.com/watch?v=Lg1QI8ZkPRU
2,Watched,I Love These Uma Musume,Starfang,2026-09-07 13:44:34,https://www.youtube.com/watch?v=5OOE4WORb5Q
3,Watched,第三方居然出了MG飛昇自由？大砲還有拉伸結構，還可以全彈髮射！ #gundam #갓건담 #...,玩具測評總動員,2026-09-07 13:43:43,https://www.youtube.com/watch?v=R9922GyZmvw
4,Watched,【#なつめぇ誕生日2026】🎂ケーキ食べようよ！【来栖夏芽/にじさんじ】,来栖 夏芽-kurusu natsume-【にじさんじ】,2026-09-07 13:35:14,https://www.youtube.com/watch?v=Ftji2LKqVXs
...,...,...,...,...,...
9940,Watched,Wonder Acute (Onsen) #ウマ娘 #umamusume #anime #...,Kidd Thunder 76,2026-07-07 17:56:08,https://www.youtube.com/watch?v=fbnlwV1jmdE
9941,Watched,I NEVER lost faith in YukaF,NiceWigg,2026-07-07 17:54:06,https://www.youtube.com/watch?v=BcbTEjpaY5M
9942,Watched,Haru Breaks The Fourth Wall #voiceacting#voice...,hapyr,2026-07-07 17:54:00,https://www.youtube.com/watch?v=qXvvSKL0hgA
9943,Watched,An Agnes Digital Interaction #animecharacter#a...,hapyr,2026-07-07 17:53:34,https://www.youtube.com/watch?v=wUfEL06bVSM


We can use the .str attribute to access all the strings in a specific row entry, and then we can use [n:n] to slice off a specific section of the string. In this context, we slice off the timezone so we can more easily convert our entry times that are currently strings into timestamp data types

In [92]:
#converted_watched_at_entries = pd.to_datetime(records_dataframe["watched_at"].str[:-4], dayfirst=True)

#records_dataframe["watched_at"] = converted_watched_at_entries

records_dataframe.head()



,action,title,channel,watched_at,video_url
0,Watched,Mario Kart 8 Deluxe Players are Going Insane r...,Shortcat,2026-09-07 13:51:45,https://www.youtube.com/watch?v=TOKdln0vkrw
1,Watched,Oops! Nintendo reveals Mario Kart 8's DEBUG mode,Beta64,2026-09-07 13:50:46,https://www.youtube.com/watch?v=Lg1QI8ZkPRU
2,Watched,I Love These Uma Musume,Starfang,2026-09-07 13:44:34,https://www.youtube.com/watch?v=5OOE4WORb5Q
3,Watched,第三方居然出了MG飛昇自由？大砲還有拉伸結構，還可以全彈髮射！ #gundam #갓건담 #...,玩具測評總動員,2026-09-07 13:43:43,https://www.youtube.com/watch?v=R9922GyZmvw
4,Watched,【#なつめぇ誕生日2026】🎂ケーキ食べようよ！【来栖夏芽/にじさんじ】,来栖 夏芽-kurusu natsume-【にじさんじ】,2026-09-07 13:35:14,https://www.youtube.com/watch?v=Ftji2LKqVXs


## Determining what is a music video or not

There are a few methods we could use to track whether a video is music related or not.
- Keeping track of a channel that is music-focused and checking if a video comes from that channel (also works for topic channels
- Checking the title of the video for certain keywords
- Using playlist data, if a video appears in a playlist, then it should be included
- Checking the metadata from the URL, either from the Takeout data or Youtube's own metadata

To do that though, we'll need to extend our DataFrame furhter to be able to include video ID as a metric for comparison

In [99]:
video_ids = records_dataframe["video_url"].str.split("v=").str[1]

video_ids.str.len().unique() #Checking that all video ids equally return 11 characters

records_dataframe["video_ids"] = video_ids

records_dataframe.head()
#records_dataframe.shape

,action,title,channel,watched_at,video_url,video_ids
0,Watched,Mario Kart 8 Deluxe Players are Going Insane r...,Shortcat,2026-09-07 13:51:45,https://www.youtube.com/watch?v=TOKdln0vkrw,TOKdln0vkrw
1,Watched,Oops! Nintendo reveals Mario Kart 8's DEBUG mode,Beta64,2026-09-07 13:50:46,https://www.youtube.com/watch?v=Lg1QI8ZkPRU,Lg1QI8ZkPRU
2,Watched,I Love These Uma Musume,Starfang,2026-09-07 13:44:34,https://www.youtube.com/watch?v=5OOE4WORb5Q,5OOE4WORb5Q
3,Watched,第三方居然出了MG飛昇自由？大砲還有拉伸結構，還可以全彈髮射！ #gundam #갓건담 #...,玩具測評總動員,2026-09-07 13:43:43,https://www.youtube.com/watch?v=R9922GyZmvw,R9922GyZmvw
4,Watched,【#なつめぇ誕生日2026】🎂ケーキ食べようよ！【来栖夏芽/にじさんじ】,来栖 夏芽-kurusu natsume-【にじさんじ】,2026-09-07 13:35:14,https://www.youtube.com/watch?v=Ftji2LKqVXs,Ftji2LKqVXs


### Reading CSV Files for Video IDs

Now we want to try converting the CSV files from the music and playlist folders from the google takeout data to act as our first basis for finding music videos.

In [100]:
music_library_csv_path = project_root / "data" / "raw" / "music library songs.csv"

music_library_dataframe = pd.read_csv(music_library_csv_path)


In [101]:
music_library_dataframe.head()


,Video ID,Song Title,Album Title,Artist Name 1,Artist Name 2
0,1aTIvok5__w,Across the Galaxies,Deep Sleep Relaxation Music,Mrm Team,NaN
1,B2i_23Ufnj0,aLIEz [mZk ver.] - aLIEz (mZk Version),&Z,SawanoHiroyuki[nZk],mizuki
2,tQRk5OFsGXo,Andromeda,Still Still Stellar,星街すいせい,NaN
3,UxG0N1Nfoq0,朝です… - Asadesu,Seishunbutayarouha Bunny Girl Senpaino Yumeomi...,fox capture plan,NaN
4,fFB88ikHOGI,Aurora: Delta Waves 2.5 Hz,"Brain Waves Music, Vol. 1 - Delta Waves",Mrm team,NaN


Now that we have a 2nd dataframe, we can start checking if there are songs in our watch history that are listed within some of our music libraries

In [102]:
library_matches = music_library_dataframe["Video ID"].isin(records_dataframe["video_ids"])

library_matches.sum()

np.int64(2)

So it seems like there are 2 instances of music in the library that were in my watch history.

While i think that this method is interesting, I'm unsure how useful it could be, at least for this specific instance of my Google takeout as it is very small and relatively recent (only going back to the 7th of July this year), which would mean cross-referencing video ids to playlists would be relatively pointless given that I've mostly switched to usingYyoutube's auto generated playlists, and that I also don't listen to those songs as often (if at all) anymore. in my opinion, I think it would be a better idea to use some of our other methods.